# Opvullen DimBlueBike en DimStation

Maar een keer runnen anders krijg je errors op de constraints in de DWH

In [ ]:
# Imports and loadin for database
import pandas as pd
from pathlib import Path
import os
import sys
from langchain_groq import ChatGroq
import requests
import re
import numpy as np

# Setup paths
ROOT = Path().resolve().parents[1] 
sys.path.append(str(ROOT))

from DWH.connection.connect import loadIN, getData, text
from data_gathering.dimLocation.initializer import getLocationKey

GROQ_API_KEY = os.getenv("groq_API")
ZIPCODE_FILE=Path.cwd().parent.parent / "data" / "zipcodes_num_nl_2025.xls" # wordt DimLocation
BLUEBIKE_OUTPUT    = ROOT / "data" / "bluebikes" / "dim_blue_bike.csv"
STATION_OUTPUT    = ROOT / "data" / "NMBS" / "dim_station.csv"

In [ ]:
llm = ChatGroq(
    temperature=0, 
    model_name="llama-3.3-70b-versatile", 
    groq_api_key=GROQ_API_KEY)

df_zip = pd.read_excel(ZIPCODE_FILE)

In [ ]:
def find_with_ai(prompt):
    try:
        return llm.invoke(prompt).content.strip()
    except Exception as e:
        print(f"AI Error: {e}")
        return None


def get_vlaanderen_data(lat, lon, locatie):
    url = f"https://geo.api.vlaanderen.be/geolocation/v4/Location?latlon={lat},{lon}"
    try:
        r = requests.get(url, timeout=5)
        res = r.json().get('LocationResult', [])
        if res:
            return str(res[0].get('Zipcode'))
    except:
        pass
    # AI Fallback voor waalse locaties
    gemeente = find_with_ai(
        f"Geef de gemeente/stad van deze locatie:\nLocatie: '{locatie}'\n"
        "Geef de Waalse versie van de gemeente.\nAntwoord enkel met de gemeente"
    )
    pc = find_with_ai(
        f"Geef de postcode van deze gemeente/stad\n{gemeente}\nAntwoord enkel met de postcode"
    )
    return str(pc)

# main
def find_gemeente_in_location(location, locaties):
    location_lower = location.lower()
    for gemeente in locaties:
        if gemeente.lower() in location_lower:
            return gemeente
    for gemeente in locaties:
    # Check elk woord/deel van de gemeentenaam apart
        for deel in re.split(r'[-\s]', gemeente):
            if deel.lower() in location_lower:
                return gemeente
    return None

# Haversine formule
def check_distance(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))

In [ ]:
print("-" * 60)
print("1. BlueBike locaties ophalen en verrijken")
print("-" * 60)

try:
    df_bluebike = pd.read_json("https://api.blue-bike.be/v4/location/website")
except Exception as e:
    print(f"API call mislukt: {e}")

df_bluebike = df_bluebike[['id', 'name', 'latitude', 'longitude' ]]
df_bluebike.rename(columns={
  'id' : 'BlueBikeStationKey',
  'name' : 'Location',
  'latitude' : 'Latitude',
  'longitude' : 'Longitude'
}, inplace=True)

print(f"Start ophalen geodata voor {len(df_bluebike)} locaties...")

dim_location = getData(query=text("""SELECT * FROM DimLocation"""))

for index, row in df_bluebike.iterrows():
    found    = False
    locatie  = row['Location']
    lat, lon = row['Latitude'], row['Longitude']

    pc = get_vlaanderen_data(lat, lon, locatie)
    print(f"[{index + 1}/{len(df_bluebike)}] {locatie}: Postcode = {pc}")

    if pc:
        try:
            match = df_zip[df_zip['Postcode'] == int(pc)]
        except (ValueError, AttributeError):
            match = pd.DataFrame()

        if len(match) > 1:
            locaties = match['Plaatsnaam'].tolist()
            gem = find_gemeente_in_location(locatie, locaties)
            if not gem:
                # AI Fallback als gemeente niet te vinden is met de gekende data
                gem = find_with_ai(
                    f"Coördinaten: ({lat}, {lon})\nPostcode = {pc}\n"
                    f"Kies de juiste gemeente uit:\n{locaties}\n"
                    "Antwoord enkel met de waarde uit de lijst, geen uitleg."
                )
            match = match[match['Plaatsnaam'] == gem]

        if len(match) == 1:
            gemeente = str(match.iloc[0]['Plaatsnaam']).title()
            # print(pc, hoofdplaats, gemeente, province)
            found = True

    if not found:
        print(f"PROBLEEM: index = {index}, pc = {pc}, loc = {locatie}")

    location_key = dim_location[(dim_location['PostalCode'] == '9870') & (dim_location['Municipality'] == 'Machelen')].iloc[0].LocationKey
    df_bluebike.at[index,'LocationKey'] = location_key
    

In [ ]:
# Converteer naar integer
df_bluebike['LocationKey'] = df_bluebike['LocationKey'].astype(int)
df_bluebike.info()

In [ ]:
print("-" * 60)
print("2. Stations ophalen met blue bike locaties binnen de 500m")
print("-" * 60)

df_stations = pd.read_csv('https://raw.githubusercontent.com/iRail/stations/refs/heads/master/stations.csv')

df_bluebike['_key'] = 1
df_stations['_key'] = 1
df_stations['URI'] = df_stations['URI'].str.replace('http://irail.be/stations/NMBS/', '', regex=False)
cross = df_bluebike.merge(df_stations, on='_key').drop('_key', axis=1)

cross['afstand_m'] = check_distance(
    cross['Latitude'], cross['Longitude'],
    cross['latitude'], cross['longitude']
)

cross['binnen_500m'] = cross['afstand_m'] <= 500

# blue bike plaatsen bij een station
df_bluebike_bij_station = (cross[cross['binnen_500m']][['BlueBikeStationKey', 'URI']]).copy()

In [ ]:
dim_station.head()

In [ ]:
print("-" * 60)
print("3. DimStation in DWH laden")
print("-" * 60)

# Filter enkel de stations die binnen 500m liggen en zorg voor juiste benaming
dim_station = cross[cross['binnen_500m']][['URI', 'name', 'latitude', 'longitude']]
dim_station.drop_duplicates(inplace=True)
dim_station.rename(columns={
        'name'     : 'StationName',
        'latitude' : 'Latitude',
        'longitude': 'Longitude',
    }, inplace=True)

print(f"Aantal unieke stations: {len(dim_station)}")
result = loadIN(df=dim_station, table='DimStation') # latitude longtitude moet erbij worden gestoken
print(f"DimStation geladen: {result}")

# Surrogate keys terugophalen uit de database na het laden
# DimStation heeft nu: StationKey (IDENTITY), URI, StationName, Latitude, Longitude
df_station_db = getData(query=text("""SELECT * FROM DimStation"""))
uri_to_surrogate = df_station_db.set_index('URI')['StationKey'].to_dict()
print(f"  {len(df_station_db)} surrogate keys opgehaald uit DimStation")

In [ ]:
print("-" * 60)
print("4. LinkedStationKey koppelen aan BlueBike locaties")
print("-" * 60)

# Station linken aan bluebike locatie
df_bluebike_bij_station['LinkedStationKey'] = df_bluebike_bij_station['URI'].map(uri_to_surrogate)

# # Verwijder LinkedStationKey en NearStation als ze al bestaan (bij heruitvoeren cel)
# df_bluebike = df_bluebike.drop(columns=['LinkedStationKey', 'NearStation'], errors='ignore')

df_bluebike = df_bluebike.merge(
    df_bluebike_bij_station[['BlueBikeStationKey', 'LinkedStationKey']],
    on='BlueBikeStationKey',
    how='left'
)

df_bluebike['NearStation'] = df_bluebike['LinkedStationKey'].notna()

print(f"BlueBike locaties met stationslink : {df_bluebike['NearStation'].sum()}")
print(f"BlueBike locaties zonder link      : {(df_bluebike['NearStation'] == False).sum()}")

In [ ]:
print("-" * 60)
print("5. DimBlueBikeStation in DWH laden")
print("-" * 60)

# Opkuisen: enkel de kolommen die in DimBlueBikeStation horen
dim_bluebike = df_bluebike[[
    'BlueBikeStationKey',
    'Location',
    'Latitude',
    'Longitude',
    'LocationKey',
    'LinkedStationKey',
    'NearStation'
]].copy()

result = loadIN(df=dim_bluebike, table='DimBlueBikeStation')
print(f"DimBlueBikeStation geladen: {result}")

In [ ]:
BLUEBIKE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
STATION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

dim_bluebike.to_csv(BLUEBIKE_OUTPUT, index=False, encoding='utf-8-sig', sep=';')
dim_station.to_csv(STATION_OUTPUT, index=False, encoding='utf-8-sig', sep=';')
print(f"\nCSV backup opgeslagen:\n  {BLUEBIKE_OUTPUT}\n  {STATION_OUTPUT}")

print("\nInitialisatie volledig voltooid!")